# Test Roman's LoRA on Kaggle (free likeness test)

Runs unattended with **Save Version → Save & Run All**.

- **Input:** the chosen LoRA `.safetensors` (attach the training notebook's output or a dataset
  with the file). Set `LORA_FILE` below to its file name; leave it empty for the final one.
- **Settings:** Accelerator **GPU T4 x2**, Internet **on**.
- **Test set:** 6 fixed prompts × LoRA strength 0.7 / 0.9 / 1.0, the same seed per prompt, Z-Image
  Turbo at 8 steps without CFG, 1024 × 1024.
- **Output (`/kaggle/working/lora_test/`):** `contact_sheet.jpg` (rows = prompts, columns =
  strengths), the 18 single images, `settings.json`, all in `lora_test.zip`.

Compare each image with Roman's photos: same face, hairline, build and age?

In [ ]:
LORA_FILE = ""  # e.g. "r0man_bg_z_image_turbo_000001500.safetensors"; empty = the final file
TRIGGER = ""  # empty = taken from the file name (the part before "_z_image")
STRENGTHS = [0.7, 0.9, 1.0]
BASE_MODEL = "Tongyi-MAI/Z-Image-Turbo"
STEPS = 8  # Z-Image Turbo
GUIDANCE = 0.0  # Turbo runs without CFG
SIZE = 1024
PROMPTS = {  # name -> prompt; {trigger} is filled in; one fixed seed each
    "office portrait": ("{trigger}, a man in a navy suit in a bright modern office, professional "
                        "portrait, soft window light, sharp focus", 1001),
    "facebook ad": ("{trigger}, a man standing on the right side of the frame in a modern tech "
                    "office with blue accent light, the left half is calm, empty dark background "
                    "for text, advertising photo", 1002),
    "outdoor": ("{trigger}, a man walking in a city street on a sunny day, smart casual jacket, "
                "natural light, photo", 1003),
    "side angle": ("{trigger}, side profile of a man looking out of a window, three-quarter view, "
                   "soft light, photo", 1004),
    "smiling": ("{trigger}, a man smiling warmly at the camera, close-up portrait, natural light",
                1005),
    "linkedin": ("{trigger}, professional LinkedIn headshot of a man, light grey background, "
                 "studio light, white shirt and dark blazer", 1006),
}
OUT = "/kaggle/working/lora_test"

In [ ]:
import json
import os
import subprocess
from pathlib import Path

import torch


def sh(command: str) -> None:
    """Run a shell command and stop the notebook if it fails (only used before training)."""
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True)


if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 x2, then Save & Run All again.")
MAJOR, MINOR = torch.cuda.get_device_capability(0)
# T4 (7.5) and P100 (6.0) have no native bf16: fp16 there, bf16 on newer GPUs.
DTYPE = "bf16" if MAJOR >= 8 else "fp16"
print(f"GPU: {torch.cuda.get_device_name(0)} (compute {MAJOR}.{MINOR}), "
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB, dtype {DTYPE}")


os.makedirs(OUT, exist_ok=True)
os.environ["HF_HOME"] = "/tmp/hf"
sh("pip install -q -U diffusers transformers accelerate safetensors peft")

In [ ]:
# Find the LoRA among the attached inputs.
files = sorted(Path("/kaggle/input").rglob("*.safetensors"))
if LORA_FILE:
    files = [path for path in files if path.name == LORA_FILE]
else:  # the final file has no step number
    finals = [path for path in files if not path.stem.rsplit("_", 1)[-1].isdigit()]
    files = finals or files[-1:]
if not files:
    raise SystemExit("No LoRA found: attach the training output and check LORA_FILE.")
LORA = files[0]
TRIGGER = TRIGGER or LORA.stem.split("_z_image")[0]
print(f"LoRA: {LORA} | trigger: {TRIGGER}")

In [ ]:
# Z-Image Turbo in diffusers; the model moves to the GPU part by part (fits 16 GB).
from diffusers import ZImagePipeline

TORCH_DTYPE = torch.bfloat16 if DTYPE == "bf16" else torch.float16
pipe = ZImagePipeline.from_pretrained(BASE_MODEL, torch_dtype=TORCH_DTYPE)
pipe.enable_model_cpu_offload()

In [ ]:
# Apply the LoRA: diffusers' loader if it accepts the file, else a direct merge into the
# transformer weights (both scale it per strength). Stops if nothing matches.
import re
from typing import Any

from safetensors.torch import load_file

METHOD = ""
try:
    pipe.load_lora_weights(str(LORA), adapter_name="roman")
    if "roman" in str(pipe.get_list_adapters()):
        METHOD = "diffusers"
except Exception as error:  # noqa: BLE001 - fall back to the direct merge
    print("diffusers could not load it:", type(error).__name__, str(error)[:200])

STATE = load_file(str(LORA)) if not METHOD else {}
MODULES = dict(pipe.transformer.named_modules())
UNDERSCORED = {name.replace(".", "_"): module for name, module in MODULES.items()}
PAIRS: list[tuple[Any, Any, Any, float]] = []  # (module, down, up, scale)
if not METHOD:
    for key, down in STATE.items():
        found = re.match(r"(.+)\.(lora_A|lora_down)\.weight$", key)
        if found is None:
            continue
        base, kind = found.group(1), found.group(2)
        up = STATE.get(f"{base}.{'lora_B' if kind == 'lora_A' else 'lora_up'}.weight")
        name = base
        for prefix in ("diffusion_model.", "transformer.", "base_model.model.", "lora_unet_"):
            name = name.removeprefix(prefix)
        module = MODULES.get(name) or UNDERSCORED.get(name)
        if up is None or module is None or not hasattr(module, "weight"):
            continue
        alpha = STATE.get(f"{base}.alpha")
        scale = float(alpha) / down.shape[0] if alpha is not None else 1.0
        PAIRS.append((module, down, up, scale))
    total = sum(1 for key in STATE if re.search(r"\.(lora_A|lora_down)\.weight$", key))
    print(f"Direct merge: {len(PAIRS)} of {total} LoRA layers matched the transformer")
    if not PAIRS or len(PAIRS) < 0.9 * total:
        print("Example LoRA keys:", list(STATE)[:5])
        print("Example model modules:", list(MODULES)[1:6])
        raise SystemExit("The LoRA does not fit this model (see the names above).")
    METHOD = "merge"
APPLIED = 0.0


def set_strength(strength: float) -> None:
    global APPLIED
    if METHOD == "diffusers":
        pipe.set_adapters(["roman"], adapter_weights=[strength])
        return
    step = strength - APPLIED
    with torch.no_grad():
        for module, down, up, scale in PAIRS:
            weight = module.weight
            delta = (up.float() @ down.float()) * (scale * step)
            weight.data += delta.to(device=weight.device, dtype=weight.dtype)
    APPLIED = strength


print("LoRA applied by:", METHOD)

In [ ]:
# The fixed test set: every prompt at every strength, same seed per prompt.
from PIL import Image, ImageDraw, ImageFont

RESULTS: dict[tuple[str, float], Path] = {}
for strength in STRENGTHS:
    set_strength(strength)
    for name, (prompt, seed) in PROMPTS.items():
        image = pipe(
            prompt=prompt.format(trigger=TRIGGER),
            height=SIZE,
            width=SIZE,
            num_inference_steps=STEPS,
            guidance_scale=GUIDANCE,
            generator=torch.Generator("cpu").manual_seed(seed),
        ).images[0]
        path = Path(OUT, f"{name.replace(' ', '_')}_s{strength:.1f}.png")
        image.save(path)
        RESULTS[(name, strength)] = path
        print("saved", path.name, flush=True)

In [ ]:
# One contact sheet (rows = prompts, columns = strengths) and one zip with everything.
import zipfile

CELL, LABEL, HEAD = 384, 170, 40
names = list(PROMPTS)
sheet = Image.new("RGB", (LABEL + CELL * len(STRENGTHS), HEAD + CELL * len(names)), "white")
draw = ImageDraw.Draw(sheet)
font = ImageFont.load_default(size=20)
for column, strength in enumerate(STRENGTHS):
    draw.text((LABEL + column * CELL + 10, 10), f"strength {strength}", fill="black", font=font)
for row, name in enumerate(names):
    draw.text((10, HEAD + row * CELL + CELL // 2), name, fill="black", font=font)
    for column, strength in enumerate(STRENGTHS):
        with Image.open(RESULTS[(name, strength)]) as image:
            sheet.paste(image.resize((CELL, CELL)), (LABEL + column * CELL, HEAD + row * CELL))
sheet.save(Path(OUT, "contact_sheet.jpg"), quality=92)
settings = {"lora": LORA.name, "trigger": TRIGGER, "method": METHOD, "strengths": STRENGTHS,
            "steps": STEPS, "guidance": GUIDANCE, "size": SIZE, "dtype": DTYPE,
            "prompts": {name: {"prompt": prompt.format(trigger=TRIGGER), "seed": seed}
                        for name, (prompt, seed) in PROMPTS.items()}}
Path(OUT, "settings.json").write_text(json.dumps(settings, indent=2))
with zipfile.ZipFile(Path(OUT, "lora_test.zip"), "w") as bundle:
    for path in sorted(Path(OUT).iterdir()):
        if path.suffix != ".zip":
            bundle.write(path, path.name)
print("Contact sheet:", Path(OUT, "contact_sheet.jpg"))
print("If he looks right: uv run ca lora set zimage", LORA.name)